# glass-0-1-6_vs_2: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('15_glass-0-1-6_vs_2_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,40.246,77.762,0.000
smote,66.600,36.809,28.754,53.333,86.571,69.952,42.127,80.119,51.705
bsmote,59.727,33.552,34.167,41.667,90.000,65.833,36.794,75.143,46.640
ros,64.950,34.159,27.262,52.500,86.857,69.679,41.670,83.167,49.554
adasyn,68.178,37.180,28.690,56.667,86.857,71.762,44.102,80.119,52.679
casb,51.260,26.867,23.274,34.167,88.571,61.369,39.484,78.524,39.063
casb_relaxed,60.989,33.904,27.357,49.167,88.000,68.583,43.085,78.548,47.446
casb_edited,67.434,34.735,26.121,56.667,84.000,70.333,38.633,78.762,51.084
casb_ros,56.804,31.655,27.817,42.500,88.857,65.679,40.861,80.476,44.230


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,99.429,49.714,34.801,72.143,0.000
smote,66.068,33.627,25.534,55.000,82.571,68.786,38.644,76.857,49.847
bsmote,60.495,30.459,24.433,45.000,84.286,64.643,34.038,74.286,45.477
ros,65.182,33.823,26.286,51.667,85.429,68.548,35.572,73.524,49.503
adasyn,65.312,34.847,27.278,51.667,85.143,68.405,38.702,73.143,50.080
casb,59.386,32.355,28.250,41.667,87.143,64.405,40.977,75.143,45.871
casb_relaxed,60.617,30.556,24.520,45.000,84.571,64.786,36.246,74.321,45.586
casb_edited,66.915,33.228,25.187,58.333,80.857,69.595,33.274,74.202,50.071
casb_ros,61.270,34.037,31.167,45.000,86.857,65.929,38.920,73.119,47.654


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,11.547,10.000,20.000,6.667,99.714,53.190,34.837,78.250,10.774
smote,31.153,22.000,38.333,16.667,95.143,55.905,35.831,78.143,26.576
bsmote,21.381,16.000,30.000,11.667,94.571,53.119,34.004,77.988,18.690
ros,21.380,16.333,33.333,11.667,95.429,53.548,35.825,78.595,18.856
adasyn,31.754,22.167,37.000,17.500,94.571,56.036,34.844,78.786,26.960
casb,26.308,19.667,38.333,14.167,96.286,55.226,35.532,78.286,22.987
casb_relaxed,36.841,26.524,47.500,20.000,96.571,58.286,35.019,77.357,31.682
casb_edited,35.919,19.762,20.000,20.000,92.857,56.429,35.873,78.333,27.840
casb_ros,21.381,16.000,30.000,11.667,96.571,54.119,36.308,79.488,18.690


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,98.286,49.143,32.999,76.095,0.000
smote,66.734,35.076,26.453,55.833,85.143,70.488,45.188,85.643,50.905
bsmote,64.019,33.752,27.536,51.667,86.857,69.262,41.746,82.452,48.885
ros,70.605,36.126,27.317,63.333,82.571,72.952,45.286,86.143,53.366
adasyn,69.790,36.187,26.022,61.667,83.714,72.690,47.230,85.429,52.988
casb,66.592,36.902,28.476,55.000,86.571,70.786,46.472,84.119,51.747
casb_relaxed,69.960,40.490,31.274,60.000,86.286,73.143,43.725,84.405,55.225
casb_edited,64.864,30.268,21.294,55.000,81.429,68.214,43.441,82.452,47.566
casb_ros,66.560,36.513,27.952,55.000,86.571,70.786,46.951,84.548,51.536


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,2.887,2.500,5.000,1.667,99.357,50.512,35.721,76.062,2.693
smote,57.639,31.878,29.769,45.208,87.357,66.283,40.447,80.190,44.758
bsmote,51.406,28.441,29.034,37.500,88.929,63.214,36.646,77.467,39.923
ros,55.529,30.110,28.550,44.792,87.571,66.182,39.588,80.357,42.820
adasyn,58.758,32.595,29.748,46.875,87.571,67.223,41.219,79.369,45.677
casb,50.887,28.948,29.583,36.250,89.643,62.946,40.616,79.018,39.917
casb_relaxed,57.102,32.868,32.663,43.542,88.857,66.199,39.519,78.658,44.985
casb_edited,58.783,29.498,23.151,47.500,84.786,66.143,37.805,78.438,44.141
casb_ros,51.504,29.551,29.234,38.542,89.714,64.128,40.760,79.408,40.527


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,67.434,adasyn,68.178,-0.744,LOSS
1,f1,casb_edited,34.735,adasyn,37.180,-2.445,LOSS
2,precision,casb_ros,27.817,bsmote,34.167,-6.349,LOSS
3,recall,casb_edited,56.667,adasyn,56.667,0.000,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,66.915,smote,66.068,0.846,WIN
1,f1,casb_ros,34.037,adasyn,34.847,-0.810,LOSS
2,precision,casb_ros,31.167,adasyn,27.278,3.889,WIN
3,recall,casb_edited,58.333,smote,55.000,3.333,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,36.841,adasyn,31.754,5.087,WIN
1,f1,casb_relaxed,26.524,adasyn,22.167,4.357,WIN
2,precision,casb_relaxed,47.500,smote,38.333,9.167,WIN
3,recall,casb_relaxed,20.000,adasyn,17.500,2.500,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,58.783,adasyn,58.758,0.025,WIN
1,f1,casb_relaxed,32.868,adasyn,32.595,0.273,WIN
2,precision,casb_relaxed,32.663,smote,29.769,2.894,WIN
3,recall,casb_edited,47.500,adasyn,46.875,0.625,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.164312,0.621429,0.000000,None
1,42,1,svm,smote,1.00,0.462910,0.200000,0.166667,0.250000,0.857143,0.553571,0.392387,0.635714,0.331455,None
2,42,1,svm,bsmote,1.00,0.478091,0.250000,0.250000,0.250000,0.914286,0.582143,0.429293,0.685714,0.364046,None
3,42,1,svm,ros,0.75,0.485504,0.285714,0.333333,0.250000,0.942857,0.596429,0.295788,0.742857,0.385609,None
4,42,1,svm,adasyn,0.75,0.478091,0.250000,0.250000,0.250000,0.914286,0.582143,0.381842,0.642857,0.364046,None
5,42,1,svm,casb,0.75,0.485504,0.285714,0.333333,0.250000,0.942857,0.596429,0.455303,0.721429,0.385609,None
6,42,1,svm,casb_relaxed,0.75,0.485504,0.285714,0.333333,0.250000,0.942857,0.596429,0.411706,0.671429,0.385609,None
7,42,1,svm,casb_edited,0.75,0.478091,0.250000,0.250000,0.250000,0.914286,0.582143,0.408460,0.671429,0.364046,None
8,42,1,svm,casb_ros,0.75,0.492805,0.333333,0.500000,0.250000,0.971429,0.610714,0.463462,0.771429,0.413069,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.326923,0.550000,0.000000,None
